# Links 
github repo: https://github.com/vildeskobba/IND320vilde 

Streamlit app: https://ind320vilde.streamlit.app/ 

Panopto_video: 


In [127]:
%pip install -q -r requirements.txt


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [200]:
import pandas as pd
import requests
import tomllib
from pymongo import MongoClient
import certifi
import numpy as np
from river import preprocessing, linear_model, compose
import geopandas as gpd
from datetime import date


In [129]:
def get_exchange_rates(url:str):
    response = requests.get(
        url,
        params = {
            "format": "sdmx-json",
            "locale": "no",
        },
        timeout = 30,
    )
    response.raise_for_status() #raise error if request was unsuccessfull 
    data = response.json()["data"] # convert json to dict

    dimensions = data["structure"]["dimensions"]
    series_dims = dimensions["series"]

    currencies = {
        d["id"]: d["values"] for d in series_dims
    }

    dates = dimensions["observation"][0]["values"]

    records = []
    for key, series in data["dataSets"][0]["series"].items():
        indices = key.split(":")
        base = currencies["BASE_CUR"][int(indices[2])]["id"]
        quote = currencies["QUOTE_CUR"][int(indices[3])]["id"]

        for date, values in series["observations"].items():
            records.append([
                dates[int(date)]["id"], base, quote, values[0]
            ])

    df = pd.DataFrame(
        records,
        columns=["TIME_PERIOD", "BASE_CUR", "QUOTE_CUR", "OBS_VALUE"]
        )

    df["TIME_PERIOD"] = pd.to_datetime(df["TIME_PERIOD"]) 

    return df
    

In [130]:
usd_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.USD.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(usd_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      USD       NOK   10.2971
1  2024-01-03      USD       NOK   10.3672
2  2024-01-04      USD       NOK   10.3027
3  2024-01-05      USD       NOK   10.3553
4  2024-01-08      USD       NOK     10.38


In [131]:
eur_rates_df = get_exchange_rates("https://data.norges-bank.no/api/data/EXR/B.EUR.NOK.SP?endPeriod=2026-11-17&format=sdmx-json&locale=no&startPeriod=2024-01-01")

print(eur_rates_df.head())

  TIME_PERIOD BASE_CUR QUOTE_CUR OBS_VALUE
0  2024-01-02      EUR       NOK   11.2815
1  2024-01-03      EUR       NOK     11.32
2  2024-01-04      EUR       NOK   11.2845
3  2024-01-05      EUR       NOK    11.309
4  2024-01-08      EUR       NOK    11.362


In [132]:


with open(".streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)

mongo_uri = secrets["MONGO_URI"]

client = MongoClient(mongo_uri, tlsCAFile=certifi.where())

db = client["power_data"]

In [ ]:
# magasin statestikk 
from utils.APIs.NVE_API import get_nve_data
magasinstatistikk_df = get_nve_data(start_date="2020.01.01", end_date="2025.12.31")
print(f"magasinstatistikk, shape: {magasinstatistikk_df.shape} \n", magasinstatistikk_df[["date", "reservoir level", "area number"]].head(4), "\n")

# entso-e 
with open(".streamlit/secrets.toml", "rb") as f:
    secrets = tomllib.load(f)
mongo_uri = secrets["MONGO_URI"]
client = MongoClient(mongo_uri, tlsCAFile=certifi.where())
db = client["power_data"]
daily_average_flows_entsoe_df = pd.DataFrame(
    list(db["daily_average_flows"].find({}, {"_id": 0}))
)
print(f"entso-e, shape: {daily_average_flows_entsoe_df.shape} \n", daily_average_flows_entsoe_df.head(4), "\n")

# openmeteo 
open_meteo_df = pd.read_parquet("data/open_meteo/NO_1.parquet")
print(f"open meteo, shape: {open_meteo_df.shape} \n", open_meteo_df.head(4), "\n")

# norges bank 
exchange_rates_df = pd.concat([eur_rates_df, usd_rates_df], ignore_index=True) 
print(f"norges bank, shape: {exchange_rates_df.shape} \n", exchange_rates_df.head(4))

magasinstatistikk, shape: (2817, 11) 
         date  reservoir level  area number
0 2020-01-05         0.647632            5
1 2020-01-05         0.534490            3
2 2020-01-05         0.614909            3
3 2020-01-05         0.640970            0 

entso-e, shape: (106, 3) 
       flow_mw  iso_year  iso_week
0  316.488444      2024        42
1  396.950804      2024        43
2  414.828923      2024        44
3  379.591970      2024        45 

open meteo, shape: (8784000, 7) 
                        time   latitude  longitude  temperature  precipitation  \
0 2024-01-01 00:00:00+01:00  61.936407  10.919136    -8.735001            0.1   
1 2024-01-01 01:00:00+01:00  61.936407  10.919136    -8.635000            0.1   
2 2024-01-01 02:00:00+01:00  61.936407  10.919136    -8.635000            0.1   
3 2024-01-01 03:00:00+01:00  61.936407  10.919136    -8.535000            0.1   

                 date                   week  
0 2024-01-01 00:00:00  2024-01-01/2024-01-07  
1 2024-01-0

time zone differences for norges bank, open-meteo, entso-e and magasin statistikk: 
I asked chat gpt, and got this answer: 
| Data source | Time convention | Resolution | Important detail |
|---|---|---|---|
| Norges Bank | CET/CEST | Daily | Exchange rates published around 16:00 local time |
| Open-Meteo | Configurable | Hourly/daily | Default is GMT/UTC; can request Europe/Oslo |
| ENTSO-E | UTC in API responses | Hourly/15-min | Daily periods follow the relevant bidding area's local calendar |
| NVE | Norwegian local calendar | Weekly | Measurements represent Sunday at 24:00, or the nearest available measurement |

it provided links as sources and i checked them all and found: 
 for norges bank, data is published daily at 16:00(assuming local time), the data shows it is collected at "ECB concertation tidspunkt 14:15 CET", central european time. 

 open meteo: i set time to Oslo-timezone in the api func. resolution is hourly, but im guessing i will use daily, calculating a daily average later in assignment. 

 entso-e: daily resolution 

 nve: weekly, confirmed on the website. 

 to be able to use them all im going to get all the time cols on the same format. oslo and brussels have no time difference, both on CET time, im not going to account for daylight savings, becuase i will use daily resolution. just for my own ease, i will rename the time col to "date" for all the dfs. 

In [213]:
magasinstatistikk_df = magasinstatistikk_df[
    magasinstatistikk_df["area type"] == "EL"
].copy() 

magasinstatistikk_df["area"] = (
    "NO_" + magasinstatistikk_df["area number"].astype(str)
)

magasinstatistikk_df = magasinstatistikk_df[[
    "year (ISO)", 
    "week (ISO)", 
    "area", 
    "reservoir level", 
    "level (TWh)", 
    "capacity (TWh)"
    ]].copy()

magasinstatistikk_df = magasinstatistikk_df[[
    "year (ISO)", "week (ISO)", "area",
    "reservoir level", "level (TWh)", "capacity (TWh)"
]].rename(columns={
    "year (ISO)": "iso_year",
    "week (ISO)": "iso_week",
    "area": "area",
    "reservoir level": "reservoir_level",
    "level (TWh)": "level_TWh",
    "capacity (TWh)": "capacity_TWh"
}).copy()


In [135]:
# Helper function: convert dates to ISO year and ISO week
def add_iso_week(df, date_col="date"):
    df = df.copy()
    dates = pd.to_datetime(df[date_col])

    iso = dates.dt.isocalendar()
    df["iso_year"] = iso["year"]
    df["iso_week"] = iso["week"]

    return df

In [136]:
# ENTSO-E: UTC -> Norwegian time
daily_average_flows_entsoe_df["date"] = (
    pd.to_datetime(
        daily_average_flows_entsoe_df["timestamp"],
        utc=True
    )
    .dt.tz_convert("Europe/Oslo")
    .dt.tz_localize(None)
)

# Aggregate to weekly averages
entsoe_weekly = (
    daily_average_flows_entsoe_df
    .groupby(pd.to_datetime(
        daily_average_flows_entsoe_df["date"]
    ).dt.to_period("W-SUN"))
    .agg({"flow_mw": "mean"})
    .reset_index(drop=True)
)

# Add ISO year/week
entsoe_weekly["date"] = (
    daily_average_flows_entsoe_df
    .groupby(pd.to_datetime(
        daily_average_flows_entsoe_df["date"]
    ).dt.to_period("W-SUN"))["date"]
    .first()
    .values
)

daily_average_flows_entsoe_df = add_iso_week(entsoe_weekly)


In [137]:
# Open-Meteo: hourly -> weekly
open_meteo_df["date"] = (
    pd.to_datetime(open_meteo_df["time"])
    .dt.tz_localize(None)
)

open_meteo_df["week"] = (
    open_meteo_df["date"].dt.to_period("W-SUN")
)

weather_weekly = (
    open_meteo_df
    .groupby(["latitude", "longitude", "week"])
    .agg({
        "temperature": "mean",
        "precipitation": "sum"
    })
    .reset_index()
)

weather_weekly["date"] = weather_weekly["week"].dt.start_time
weather_weekly = add_iso_week(weather_weekly)

In [138]:
# Norges Bank: daily -> weekly
exchange_rates_df["date"] = pd.to_datetime(
    exchange_rates_df["TIME_PERIOD"]
)

exchange_rates_df["OBS_VALUE"] = pd.to_numeric(
    exchange_rates_df["OBS_VALUE"],
    errors="coerce"
)

exchange_weekly = (
    exchange_rates_df
    .groupby(
        exchange_rates_df["date"].dt.to_period("W-SUN")
    )
    .agg({"OBS_VALUE": "mean"})
    .reset_index()
)

exchange_weekly["date"] = (
    exchange_rates_df
    .groupby(exchange_rates_df["date"].dt.to_period("W-SUN"))["date"]
    .first()
    .values
)

exchange_rates_df = add_iso_week(exchange_weekly)

In [139]:
daily_average_flows_entsoe_df = daily_average_flows_entsoe_df.drop(columns="date")
weather_weekly = weather_weekly.drop(columns=["date", "week"])
exchange_rates_df = exchange_rates_df.drop(columns="date")

In [140]:
# Load reservoir area polygons
reservoir_areas = gpd.read_file(
    "data/nve/Budområder/Budområder.shp"
).to_crs(epsg=4326)

# Convert weather coordinates to GeoDataFrame
weather_gdf = gpd.GeoDataFrame(
    weather_weekly.copy(),
    geometry=gpd.points_from_xy(
        weather_weekly["longitude"],
        weather_weekly["latitude"]
    ),
    crs="EPSG:4326"
)

# Spatial join: assign each coordinate to a reservoir area
weather_gdf = gpd.sjoin(
    weather_gdf,
    reservoir_areas[["budomr", "geometry"]],
    how="left",
    predicate="within"
)

# Convert back to regular DataFrame
weather_weekly = pd.DataFrame(
    weather_gdf.drop(columns=["geometry", "index_right"])
).rename(columns={"budomr": "area"})
weather_weekly["area"] = (
    weather_weekly["area"]
    .astype("string")
    .str.replace(" ", "_", regex=False)
)

# average in the area 
weather_weekly = (
    weather_weekly
    .groupby(["area", "iso_year", "iso_week"], as_index=False)
    [["temperature", "precipitation"]]
    .mean()
)

In [214]:
for name, df in {
    "Magasin": magasinstatistikk_df,
    "ENTSO-E": daily_average_flows_entsoe_df,
    "Weather": weather_weekly,
    "Exchange": exchange_rates_df
}.items():
    print(f"\n{name}, shape: {df.shape}:")
    print(df.head(4))


Magasin, shape: (1565, 6):
   iso_year  iso_week  area  reservoir_level  level_TWh  capacity_TWh
0      2020         1  NO_5         0.647632  11.262702     17.390587
2      2020         1  NO_3         0.614909   5.485566      8.920932
4      2020         1  NO_2         0.715268  24.350306     34.043590
6      2020         1  NO_1         0.676240   4.059648      6.003264

ENTSO-E, shape: (106, 3):
      flow_mw  iso_year  iso_week
0  316.488444      2024        42
1  396.950804      2024        43
2  414.828923      2024        44
3  379.591970      2024        45

Weather, shape: (53, 5):
   area  iso_year  iso_week  temperature  precipitation
0  NO_1      2024         1   -16.577379         7.8036
1  NO_1      2024         2    -5.979449         5.9157
2  NO_1      2024         3   -14.830154        10.3622
3  NO_1      2024         4    -1.243873        29.8647

Exchange, shape: (151, 3):
   OBS_VALUE  iso_year  iso_week
0  10.814663      2024         1
1  10.821460      2024   

I can see that i need more power flow data, want entire 2024, so i will use api to retrieve more. I decided to let the code where i fetch from MongoDB database remain, because i assume using this database is implied part of assignment. 

In [147]:
import importlib
import utils.APIs.ENTSOE_API as entsoe_api

importlib.reload(entsoe_api)

get_entsoe_data = entsoe_api.get_entsoe_data

In [ ]:
#todo 

ENTSOE_API_TOKEN = secrets["ENTSOE_API_TOKEN"] 

from utils.APIs.ENTSOE_API import get_entsoe_data 

borders = [
    ("NO_1", "SE_3"),
    ("NO_2", "DK_1"),
    ("NO_2", "NL"),
    ("NO_2", "DE_LU"),
    ("NO_2", "GB"),
    ("NO_3", "SE_2"),
    ("NO_4", "SE_2"),
]

start = pd.Timestamp("2024-01-01", tz="Europe/Brussels")
end = pd.Timestamp("2025-01-01", tz="Europe/Brussels")
dfs = []
for a, b in borders:
    border_df = get_entsoe_data(ENTSOE_API_TOKEN, zone_a=a, zone_b=b, start=start, end=end)
    dfs.append(border_df)
    print(f"\n---succesfully downloaded and added {a}->{b} flows.\n")

In [ ]:
entso_e_dfs = {
    'NO1_SE3' : "data/entso_e_2024/no1_se3.parquet",
    'NO2_DELU': "data/entso_e_2024/no2_delu.parquet",
    'NO2_DK1': "data/entso_e_2024/no2_dk1.parquet",
    'NO2_GB': "data/entso_e_2024/no2_gb.parquet",
    'NO2_NL': "data/entso_e_2024/no2_nl.parquet",
    'NO3_SE2': "data/entso_e_2024/no3_se2.parquet",
    'NO4_SE2': "data/entso_e_2024/no4_se2.parquet",
}

for k, v in entso_e_dfs.items(): 
    df = pd.read_parquet(v)

    print(k, "\n", df.head(), "\n\n")

NO1_SE3 
                   timestamp from_zone to_zone  flow_MW
0 2024-01-01 00:00:00+01:00      NO_1    SE_3     0.00
1 2024-01-01 00:00:00+01:00      SE_3    NO_1  1655.69
2 2024-01-01 01:00:00+01:00      SE_3    NO_1  1822.15
3 2024-01-01 01:00:00+01:00      NO_1    SE_3     0.00
4 2024-01-01 02:00:00+01:00      NO_1    SE_3     0.00 


NO2_DELU 
                   timestamp from_zone to_zone  flow_MW
0 2024-01-01 00:00:00+01:00      NO_2   DE_LU     0.00
1 2024-01-01 00:00:00+01:00     DE_LU    NO_2  1392.58
2 2024-01-01 00:15:00+01:00     DE_LU    NO_2  1400.57
3 2024-01-01 00:15:00+01:00      NO_2   DE_LU     0.00
4 2024-01-01 00:30:00+01:00      NO_2   DE_LU     0.00 


NO2_DK1 
                   timestamp from_zone to_zone  flow_MW
0 2024-01-01 00:00:00+01:00      NO_2    DK_1     0.00
1 2024-01-01 00:00:00+01:00      DK_1    NO_2  1637.52
2 2024-01-01 01:00:00+01:00      DK_1    NO_2  1637.56
3 2024-01-01 01:00:00+01:00      NO_2    DK_1     0.00
4 2024-01-01 02:00:00+01:00 

In [188]:
collection = db["daily_average_flows"]

n_rows = collection.count_documents({})

print(f"Number of rows: {n_rows}")

Number of rows: 14626


In [187]:
collection = db["daily_average_flows"]

for name, path in entso_e_dfs.items():
    df = pd.read_parquet(path)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df["timestamp"] = df["timestamp"].dt.normalize()

    daily_avg = (
        df.groupby(["timestamp", "from_zone", "to_zone"])["flow_MW"]
        .mean()
        .reset_index()
    )

    daily_avg = daily_avg.rename(columns={"flow_MW": "flow_mw"})

    records = daily_avg.to_dict("records")

    if records:
        collection.insert_many(records)

    print(f"{name}: inserted {len(records)} rows")

NO1_SE3: inserted 732 rows
NO2_DELU: inserted 732 rows
NO2_DK1: inserted 732 rows
NO2_GB: inserted 732 rows
NO2_NL: inserted 366 rows
NO3_SE2: inserted 732 rows
NO4_SE2: inserted 366 rows


In [189]:
collection = db["daily_average_flows"]

# Create backup
collection.aggregate([
    {"$match": {}},
    {"$out": "daily_average_flows_backup"}
])

print("Backup created!")

Backup created!


In [191]:
print("Original:", db["daily_average_flows"].count_documents({}))
print("Backup:", db["daily_average_flows_backup"].count_documents({}))

# Original
df_original = pd.DataFrame(
    list(db["daily_average_flows"].find({}, {"_id": 0}).limit(5))
)

# Backup
df_backup = pd.DataFrame(
    list(db["daily_average_flows_backup"].find({}, {"_id": 0}).limit(5))
)

print("ORIGINAL:")
display(df_original)

print("BACKUP:")
display(df_backup)

Original: 14626
Backup: 14626
ORIGINAL:


,timestamp,from_zone,to_zone,flow_mw
0,2024-10-18,SE_3,NO_1,849.540
1,2024-10-18,NO_1,SE_3,0.000
2,2024-10-19,NO_2,DK_1,1631.475
3,2024-10-19,NO_3,SE_2,0.000
4,2024-10-19,NO_2,DE_LU,924.840


BACKUP:


,timestamp,from_zone,to_zone,flow_mw
0,2024-10-18,SE_3,NO_1,849.540
1,2024-10-18,NO_1,SE_3,0.000
2,2024-10-19,NO_2,DK_1,1631.475
3,2024-10-19,NO_3,SE_2,0.000
4,2024-10-19,NO_2,DE_LU,924.840


In [193]:
entso_e_df = pd.DataFrame(
    list(db["daily_average_flows"].find({}, {"_id": 0}))
)

print(entso_e_df.head())

   timestamp from_zone to_zone   flow_mw
0 2024-10-18      SE_3    NO_1   849.540
1 2024-10-18      NO_1    SE_3     0.000
2 2024-10-19      NO_2    DK_1  1631.475
3 2024-10-19      NO_3    SE_2     0.000
4 2024-10-19      NO_2   DE_LU   924.840


In [ ]:
entso_e_df = pd.DataFrame(
    list(db["daily_average_flows"].find({}, {"_id": 0}))
)

entso_e_df['date'] = pd.to_datetime(entso_e_df['timestamp'])
entso_e_df["week_start"] = (
    entso_e_df["date"]
    .dt.to_period("W-SUN")
    .dt.start_time
)

# Aggregate to weekly averages
entso_e_weekly_df = (
    entso_e_df
    .groupby([
        "week_start",
        "from_zone",
        "to_zone"
    ])
    .agg({"flow_mw": "mean"})
    .reset_index()
)

entso_e_weekly_df = add_iso_week(entso_e_weekly_df, date_col="week_start")
entso_e_weekly_df = entso_e_weekly_df.drop(columns="week_start")
print(entso_e_weekly_df.head(10))

  from_zone to_zone      flow_mw  iso_year  iso_week
0     DE_LU    NO_2  1401.286771      2023        52
1      DK_1    NO_2  1505.306250      2023        52
2        GB    NO_2   446.684167      2023        52
3      NO_1    SE_3     0.000000      2023        52
4      NO_2   DE_LU     0.000000      2023        52


In [215]:

for name, df in {
    "Magasin": magasinstatistikk_df,
    "ENTSO-E": entso_e_weekly_df,
    "Weather": weather_weekly,
    "Exchange": exchange_rates_df
}.items():
    print(f"\n{name}, shape: {df.shape}:")
    print(df.head(4))


Magasin, shape: (1565, 6):
   iso_year  iso_week  area  reservoir_level  level_TWh  capacity_TWh
0      2020         1  NO_5         0.647632  11.262702     17.390587
2      2020         1  NO_3         0.614909   5.485566      8.920932
4      2020         1  NO_2         0.715268  24.350306     34.043590
6      2020         1  NO_1         0.676240   4.059648      6.003264

ENTSO-E, shape: (1986, 5):
  from_zone to_zone      flow_mw  iso_year  iso_week
0     DE_LU    NO_2  1401.286771      2023        52
1      DK_1    NO_2  1505.306250      2023        52
2        GB    NO_2   446.684167      2023        52
3      NO_1    SE_3     0.000000      2023        52

Weather, shape: (53, 5):
   area  iso_year  iso_week  temperature  precipitation
0  NO_1      2024         1   -16.577379         7.8036
1  NO_1      2024         2    -5.979449         5.9157
2  NO_1      2024         3   -14.830154        10.3622
3  NO_1      2024         4    -1.243873        29.8647

Exchange, shape: (151,

In [241]:
def forecast_reservoir(
    magasin_df,
    entso_df=None,
    weather_df=None,
    exchange_df=None,
    area="NO_1",
    model_type="sarimax",
    history_weeks=200,
    horizon_weeks=4,
    data_sources=("weather", "entso", "exchange"),
    target="reservoir_level",
    model_params=None
):
    """
    Forecast reservoir filling for one Norwegian region.

    Parameters
    ----------
    magasin_df : DataFrame
        Magasinstatistikk data.
    entso_df : DataFrame
        ENTSO-E electricity flow data.
    weather_df : DataFrame
        Weekly weather data.
    exchange_df : DataFrame
        Weekly exchange rate data.
    area : str
        Norwegian reservoir area (NO_1 to NO_5).
    model_type : str
        "river" or "sarimax".
    history_weeks : int
        Number of historical weeks used for training.
    horizon_weeks : int
        Number of weeks to forecast.
    data_sources : tuple
        External data sources to include.
    target : str
        "reservoir_level" or "level_TWh".
    model_params : dict
        Model-specific parameters.

    Returns
    -------
    DataFrame
        Weekly reservoir filling predictions.
    """

    if model_params is None:
        model_params = {}

    # --------------------------------------------------
    # 1. Prepare weekly reservoir data
    # --------------------------------------------------

    df = magasin_df[magasin_df["area"] == area].copy()

    df["date"] = [
        date.fromisocalendar(int(y), int(w), 1)
        for y, w in zip(df["iso_year"], df["iso_week"])
    ]

    df["date"] = pd.to_datetime(df["date"])

    df = df.set_index("date").sort_index()

    df = df[[target]].copy()

    # --------------------------------------------------
    # 2. Prepare optional external data sources
    # --------------------------------------------------

    if "weather" in data_sources and weather_df is not None:

        weather = weather_df[
            weather_df["area"] == area
        ].copy()

        weather["date"] = pd.to_datetime([
            date.fromisocalendar(int(y), int(w), 1)
            for y, w in zip(
                weather["iso_year"], weather["iso_week"]
            )
        ])

        weather = weather.set_index("date").sort_index()

        weather = weather[
            ["temperature", "precipitation"]
        ]

        df = df.join(weather, how="left")

    if "entso" in data_sources and entso_df is not None:

        entso = entso_df.copy()

        entso["date"] = pd.to_datetime([
            date.fromisocalendar(int(y), int(w), 1)
            for y, w in zip(
                entso["iso_year"], entso["iso_week"]
            )
        ])

        incoming = (
            entso[entso["to_zone"] == area]
            .groupby("date")["flow_mw"]
            .sum()
            .rename("incoming_flow")
        )

        outgoing = (
            entso[entso["from_zone"] == area]
            .groupby("date")["flow_mw"]
            .sum()
            .rename("outgoing_flow")
        )

        flows = pd.concat(
            [incoming, outgoing], axis=1
        ).fillna(0)

        flows["net_flow"] = (
            flows["incoming_flow"] -
            flows["outgoing_flow"]
        )

        df = df.join(flows, how="left")

    if "exchange" in data_sources and exchange_df is not None:

        exchange = exchange_df.copy()

        exchange["date"] = pd.to_datetime([
            date.fromisocalendar(int(y), int(w), 1)
            for y, w in zip(
                exchange["iso_year"], exchange["iso_week"]
            )
        ])

        exchange = exchange.set_index("date").sort_index()

        df = df.join(
            exchange[["OBS_VALUE"]],
            how="left"
        )

    # --------------------------------------------------
    # 3. Create continuous weekly time series
    # --------------------------------------------------

    df = df[~df.index.duplicated(keep="last")]

    weekly_index = pd.date_range(
        start=df.index.min(),
        end=df.index.max(),
        freq="W-MON"
    )

    df = df.reindex(weekly_index)

    # Avoid using future information when filling gaps.
    external_cols = [
        c for c in df.columns if c != target
    ]

    if external_cols:
        df[external_cols] = (
            df[external_cols]
            .ffill()
            .fillna(0)
        )

    df = df.dropna(subset=[target])

    # Use only the requested amount of history.
    df = df.tail(history_weeks)

    if len(df) < 12:
        raise ValueError(
            "At least 12 historical observations are required."
        )

    # --------------------------------------------------
    # 4. River lag-based machine learning
    # --------------------------------------------------

    if model_type.lower() == "river":

        from river import preprocessing, linear_model

        l2 = model_params.get("l2", 0.01)

        model = (
            preprocessing.StandardScaler()
            | linear_model.LinearRegression(l2=l2)
        )

        external_cols = [
            c for c in df.columns if c != target
        ]

        values = df[target].astype(float).tolist()

        def create_features(history, external, week):

            features = {
                "lag_1": history[-1],
                "lag_2": history[-2],
                "lag_3": history[-3],
                "lag_4": history[-4],
                "mean_4": np.mean(history[-4:]),
                "mean_13": np.mean(history[-13:])
                if len(history) >= 13
                else np.mean(history),
                "week_sin": np.sin(2 * np.pi * week / 52),
                "week_cos": np.cos(2 * np.pi * week / 52)
            }

            if len(history) >= 52:
                features["lag_52"] = history[-52]

            for col in external_cols:
                features[col] = float(external[col])

            return features

        # Train sequentially using past observations.
        for i in range(13, len(df)):

            history = values[:i]

            features = create_features(
                history,
                df.iloc[i][external_cols],
                df.index[i].isocalendar().week
            )

            model.learn_one(
                features,
                values[i]
            )

        # Recursive multi-step forecasting.
        history = values.copy()
        last_external = (
            df.iloc[-1][external_cols]
            if external_cols else pd.Series(dtype=float)
        )

        predictions = []

        for step in range(1, horizon_weeks + 1):

            future_date = (
                df.index[-1] +
                pd.Timedelta(weeks=step)
            )

            features = create_features(
                history,
                last_external,
                future_date.isocalendar().week
            )

            prediction = model.predict_one(features)

            # Reservoir filling cannot be negative.
            prediction = max(0, prediction)

            # Keep the predicted value for future lags.
            history.append(prediction)
            predictions.append(prediction)

    # --------------------------------------------------
    # 5. SARIMAX alternative
    # --------------------------------------------------

    elif model_type.lower() == "sarimax":

        from statsmodels.tsa.statespace.sarimax import SARIMAX

        order = model_params.get(
            "order", (1, 1, 1)
        )

        seasonal_order = model_params.get(
            "seasonal_order", (0, 0, 0, 0)
        )

        external_cols = [
            c for c in df.columns if c != target
        ]

        y = df[target].astype(float)

        exog = df[external_cols] if external_cols else None

        future_exog = None

        if exog is not None:
            future_exog = pd.DataFrame(
                np.repeat(
                    exog.iloc[[-1]].values,
                    horizon_weeks,
                    axis=0
                ),
                columns=external_cols
            )

        model = SARIMAX(
            y,
            exog=exog,
            order=order,
            seasonal_order=seasonal_order,
            enforce_stationarity=False,
            enforce_invertibility=False
        )

        fitted_model = model.fit(
            disp=False,
            maxiter=2000,
            method="powell"
        )

        if not fitted_model.mle_retvals["converged"]:
            print("WARNING: SARIMAX did not converge")

        predictions = fitted_model.forecast(
            steps=horizon_weeks,
            exog=future_exog
        ).tolist()

        predictions = [
            max(0, float(p)) for p in predictions
        ]

    else:
        raise ValueError(
            "model_type must be 'river' or 'sarimax'."
        )

    # --------------------------------------------------
    # 6. Return forecast DataFrame
    # --------------------------------------------------

    future_dates = pd.date_range(
        start=df.index[-1] + pd.Timedelta(weeks=1),
        periods=horizon_weeks,
        freq="W-MON"
    )

    result = pd.DataFrame({
        "date": future_dates,
        "iso_year": [
            d.isocalendar().year for d in future_dates
        ],
        "iso_week": [
            d.isocalendar().week for d in future_dates
        ],
        "area": area,
        "predicted_filling": predictions
    })

    if target == "reservoir_level":
        result["predicted_filling"] = (
            result["predicted_filling"].clip(0, 1)
        )

    return result


In [222]:
forecast = forecast_reservoir(
    magasin_df=magasinstatistikk_df,
    entso_df=entso_e_weekly_df,
    weather_df=weather_weekly,
    exchange_df=exchange_rates_df,

    area="NO_1",
    model_type="river",

    history_weeks=104,
    horizon_weeks=8,

    data_sources=(
        "weather",
        "entso",
        "exchange"
    ),

    target="level_TWh",

    model_params={
        "l2": 0.01
    }
)

print(forecast)

        date  iso_year  iso_week  area  predicted_filling
0 2025-12-29      2026         1  NO_1           5.120766
1 2026-01-05      2026         2  NO_1           5.089693
2 2026-01-12      2026         3  NO_1           5.072370
3 2026-01-19      2026         4  NO_1           5.067049
4 2026-01-26      2026         5  NO_1           5.019218
5 2026-02-02      2026         6  NO_1           5.025318
6 2026-02-09      2026         7  NO_1           5.035702
7 2026-02-16      2026         8  NO_1           5.041734


In [223]:
forecast = forecast_reservoir(
    magasin_df=magasinstatistikk_df,
    entso_df=entso_e_weekly_df,
    weather_df=weather_weekly,
    exchange_df=exchange_rates_df,

    area="NO_1",
    model_type="sarimax",

    history_weeks=104,
    horizon_weeks=6,

    data_sources=("weather", "entso", "exchange"),

    target="reservoir_level",

    model_params={
        "order": (1, 1, 1),
        "seasonal_order": (0, 0, 0, 0)
    }
)

print(forecast)

        date  iso_year  iso_week  area  predicted_filling
0 2025-12-29      2026         1  NO_1           0.684971
1 2026-01-05      2026         2  NO_1           0.659453
2 2026-01-12      2026         3  NO_1           0.640075
3 2026-01-19      2026         4  NO_1           0.625359
4 2026-01-26      2026         5  NO_1           0.614185
5 2026-02-02      2026         6  NO_1           0.605700


/Users/vildeskobba/Desktop/IND320/IND320_vilde/.venv/lib/python3.13/site-packages/statsmodels/tsa/statespace/mlemodel.py:737: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  mlefit = super().fit(


In [ ]:

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd


def evaluate_forecast(
    magasin_df,
    entso_df=None,
    weather_df=None,
    exchange_df=None,
    area="NO_1",
    model_type="river",
    history_weeks=52,
    horizon_weeks=6,
    data_sources=("weather", "entso", "exchange"),
    target="reservoir_level",
    model_params=None,
    test_windows=4
):

    df = magasin_df[
        magasin_df["area"] == area
    ].copy()

    df["date"] = pd.to_datetime([
        pd.Timestamp(
            date.fromisocalendar(int(y), int(w), 1)
        )
        for y, w in zip(df["iso_year"], df["iso_week"])
    ])

    df = df.sort_values("date").reset_index(drop=True)

    errors = []

    # Expanding chronological training windows.
    for window in range(test_windows):

        test_end = len(df) - window * horizon_weeks

        test_start = test_end - horizon_weeks

        train_end = test_start

        if train_end < 13:
            continue

        train = df.iloc[:train_end]

        actual = df.iloc[
            test_start:test_end
        ][target].values

        if len(actual) != horizon_weeks:
            continue

        def subset_source(source):

            if source is None:
                return None

            source_dates = pd.to_datetime([
                date.fromisocalendar(int(y), int(w), 1)
                for y, w in zip(
                    source["iso_year"],
                    source["iso_week"]
                )
            ])

            return source.loc[
                source_dates <= train["date"].max()
            ].copy()

        prediction = forecast_reservoir(
            magasin_df=train,
            entso_df=subset_source(entso_df),
            weather_df=subset_source(weather_df),
            exchange_df=subset_source(exchange_df),

            area=area,
            model_type=model_type,
            history_weeks=history_weeks,
            horizon_weeks=horizon_weeks,
            data_sources=data_sources,
            target=target,
            model_params=model_params
        )

        predicted = prediction[
            "predicted_filling"
        ].values

        errors.append({
            "MAE": mean_absolute_error(
                actual, predicted
            ),
            "RMSE": np.sqrt(
                mean_squared_error(actual, predicted)
            )
        })

    if not errors:
        raise ValueError(
            "Not enough historical observations for backtesting."
        )

    return pd.DataFrame(errors)


In [239]:
settings = [
    ("river", 104, 4, ()),
    ("river", 104, 4, ("weather", "entso", "exchange")),
    ("river", 52, 8, ("weather",)),
    ("sarimax", 156, 4, ("weather", "entso", "exchange")),
    ("sarimax", 200, 4, ("weather", "entso", "exchange")),
    ("sarimax", 52, 8, ("weather", "entso", "exchange"))
]

results = []

for model, history, horizon, sources in settings:

    scores = evaluate_forecast(
        magasin_df=magasinstatistikk_df,
        entso_df=entso_e_weekly_df,
        weather_df=weather_weekly,
        exchange_df=exchange_rates_df,
        area="NO_1",
        model_type=model,
        history_weeks=history,
        horizon_weeks=horizon,
        data_sources=sources,
        target="reservoir_level",
        model_params={
            "order": (1, 1, 1),
            "seasonal_order": (0, 0, 0, 0)
        }
    )

    results.append({
        "model": model,
        "history": history,
        "horizon": horizon,
        "sources": str(sources),
        "MAE": scores["MAE"].mean(),
        "RMSE": scores["RMSE"].mean()
    })

results_df = pd.DataFrame(results)

print(results_df.sort_values("MAE"))

     model  history  horizon                           sources       MAE  \
4  sarimax      200        4  ('weather', 'entso', 'exchange')  0.022320   
3  sarimax      156        4  ('weather', 'entso', 'exchange')  0.022349   
5  sarimax       52        8  ('weather', 'entso', 'exchange')  0.050432   
1    river      104        4  ('weather', 'entso', 'exchange')  0.206020   
0    river      104        4                                ()  0.419244   
2    river       52        8                      ('weather',)  0.432788   

       RMSE  
4  0.027605  
3  0.027640  
5  0.060399  
1  0.215991  
0  0.441330  
2  0.457664  


the sarimax model training on 200 weeks on weather, powerflow and exchange seems to work best. 